# Edge - Extract Datasources and Build Edges

This notebook extracts datasource references and materializes lineage edges from the bronze and silver tables.

In [ ]:
import time
import pandas as pd
import re
from pyspark.sql.types import StringType, StructField, StructType

_phase_timers = {}
_table_cache = {}

def start_phase(name):
    _phase_timers[name] = time.perf_counter()
    print(f"[timer] start {name}")

def end_phase(name):
    start = _phase_timers.get(name)
    if start is None:
        print(f"[timer] {name}: missing start")
        return
    elapsed = time.perf_counter() - start
    print(f"[timer] {name}: {elapsed:.2f}s")

def load_table(table_name):
    try:
        return spark.table(table_name).toPandas()
    except Exception:
        return pd.DataFrame()

def get_table(table_name):
    if table_name not in _table_cache:
        table_start = time.perf_counter()
        _table_cache[table_name] = load_table(table_name)
        print(f"[timer] load_table:{table_name}: {time.perf_counter() - table_start:.2f}s")
    return _table_cache[table_name]

def ensure_columns(df, required_columns):
    result = df.copy() if hasattr(df, 'copy') else pd.DataFrame(df)
    if result.empty and len(result.columns) == 0:
        return pd.DataFrame(columns=required_columns)
    for column_name in required_columns:
        if column_name not in result.columns:
            result[column_name] = None
    return result

def normalize_for_spark_write(df, required_columns=None):
    final_df = ensure_columns(df, required_columns or []) if required_columns else (df.copy() if hasattr(df, 'copy') else pd.DataFrame(df))
    if final_df.empty and len(final_df.columns) == 0:
        return pd.DataFrame(columns=required_columns or [])
    for column_name in final_df.columns:
        final_df[column_name] = final_df[column_name].map(lambda value: None if pd.isna(value) or value == 'nan' else str(value))
    if required_columns:
        final_df = final_df[required_columns]
    return final_df

def write_delta_table(df, table_name):
    final_df = normalize_for_spark_write(df)
    if final_df.empty and len(final_df.columns) == 0:
        return
    schema = StructType([StructField(column_name, StringType(), True) for column_name in final_df.columns])
    spark.createDataFrame(final_df, schema=schema).write.mode('overwrite').option('overwriteSchema', 'true').format('delta').saveAsTable(table_name)

StatementMeta(, 04feeb51-f3ac-4f91-ad25-85e161dd4f5b, 5, Finished, Available, Finished, False)

In [ ]:
table_names = [
    't_dataset_columns',
    't_dataset_tables',
    't_dataset_measures',
    't_dataset_relations',
    't_dataset_dependencies',
    't_dataset_partitions',
    't_report_metadata',
    't_report_pages',
    't_report_visuals',
    't_report_semantic_objects',
    't_lakehouse_tables',
    't_lakehouse_columns',
    't_lakehouse_shortcuts',
    't_warehouse_tables',
    't_warehouse_columns',
    't_direct_lake_sources',
    't_lakehouses_meta',
    'v_nodes'
]

print(f"Registered tables for lazy loading: {', '.join(table_names)}")
print('Tables will load on first use in datasource/edge phases.')

StatementMeta(, 04feeb51-f3ac-4f91-ad25-85e161dd4f5b, 6, Finished, Available, Finished, False)

Loaded tables: t_dataset_columns, t_dataset_tables, t_dataset_measures, t_dataset_relations, t_dataset_dependencies, t_dataset_partitions, t_report_metadata, t_report_pages, t_report_visuals, t_report_semantic_objects, t_lakehouse_tables, t_lakehouse_columns, t_lakehouse_shortcuts, t_warehouse_tables, t_warehouse_columns, t_direct_lake_sources, t_lakehouses_meta, v_nodes


In [3]:
def create_object_key_from_type(obj_type, table_name, object_name, dataset_id):
    """
    Creates an object key based on object type, matching the PK logic from node creation.
    
    Parameters:
    -----------
    obj_type : str
        Type of object ('table', 'column', 'measure', etc.)
    table_name : str
        Name of the table
    object_name : str
        Name of the object (column name, measure name, etc.)
    dataset_id : str
        Dataset ID
    
    Returns:
    --------
    str : The constructed key matching node primary key patterns
    """
    obj_type_lower = str(obj_type).lower().strip()
    
    # Match the primary key patterns from Lineage_Silver_Build_Node_View
    if  'table' in obj_type_lower:
        # table_pk = table_name + '-' + dataset_id
        return f"{table_name}-{dataset_id}"
    
    elif  'column' in obj_type_lower:
        # column_pk = table_name + '-' + column_name + '-' + dataset_id
        return f"{table_name}-{object_name}-{dataset_id}"
    
    elif 'measure' in obj_type_lower:
        # measure_pk = table_name + '-' + measure_name + '-' + dataset_id
        return f"{table_name}-{object_name}-{dataset_id}"
    
    else:
        # Default pattern for other object types
        return f"{table_name}-{object_name}-{dataset_id}"


def add_dependency_foreign_keys(df):
    """
    Adds foreign key columns to the dependency dataframe.
    Creates 'object_key' and 'referenced_object_key' columns that match 
    the primary keys used in the node table.
    
    Parameters:
    -----------
    df : pd.DataFrame
        Dependency dataframe with columns like:
        - object_type, table_name, object_name, dataset_id
        - referenced_object_type, referenced_table_name, referenced_object_name
    
    Returns:
    --------
    pd.DataFrame : DataFrame with added foreign key columns
    """
    if df is None or df.empty:
        return pd.DataFrame()
    
    result = df.copy()
    
    # Create foreign key for the source object
    result['object_key'] = result.apply(
        lambda row: create_object_key_from_type(
            row.get('object_type', ''),
            row.get('table_name', ''),
            row.get('object_name', ''),
            row.get('dataset_id', '')
        ), 
        axis=1
    )
    
    # Create foreign key for the referenced object
    result['referenced_object_key'] = result.apply(
        lambda row: create_object_key_from_type(
            row.get('referenced_object_type', ''),
            row.get('referenced_table', ''),
            row.get('referenced_object', ''),
            row.get('dataset_id', '')
        ), 
        axis=1
    )
    
    return result


def add_report_object_foreign_keys(df, report_metadata_df):
    """
    Adds foreign key columns to the report semantic objects dataframe.
    Looks up dataset_id from report_metadata table using report_id.
    
    Parameters:
    -----------
    df : pd.DataFrame
        Report semantic objects dataframe with columns:
        - report_id, table_name, object_name, object_type
    report_metadata_df : pd.DataFrame
        Report metadata table containing report_id and dataset_id mapping
    
    Returns:
    --------
    pd.DataFrame : DataFrame with added object_key and dataset_id columns
    """
    if df is None or df.empty:
        return pd.DataFrame()
    
    result = df.copy()
    
    # Lookup dataset_id from report_metadata
    if not report_metadata_df.empty and 'report_id' in report_metadata_df.columns and 'dataset_id' in report_metadata_df.columns:
        report_dataset_lookup = report_metadata_df[['report_id', 'dataset_id']].drop_duplicates()
        result = result.merge(report_dataset_lookup, on='report_id', how='left', suffixes=('', '_from_report'))
        # If dataset_id already exists, keep the merged one
        if 'dataset_id_from_report' in result.columns:
            result['dataset_id'] = result['dataset_id_from_report'].fillna(result.get('dataset_id', ''))
            result = result.drop(columns=['dataset_id_from_report'])
    
    # Create foreign key for the semantic model object
    result['object_key'] = result.apply(
        lambda row: create_object_key_from_type(
            row.get('object_type', ''),
            row.get('table_name', ''),
            row.get('object_name', ''),
            row.get('dataset_id', '')
        ), 
        axis=1
    )
    
    return result

StatementMeta(, 04feeb51-f3ac-4f91-ad25-85e161dd4f5b, 7, Finished, Available, Finished, False)

In [ ]:
def normalize_text(value):
    if value is None:
        return None
    text = str(value).strip()
    if not text or text.lower() == 'nan':
        return None
    return text

def parse_source_expression(value):
    text = normalize_text(value)
    if not text:
        return None
    match = re.search(r'Source\s*=\s*(.+)', text)
    if match:
        return match.group(1).strip()
    return text

start_phase('build_dataset_datasources')
df_dataset_datasources = pd.DataFrame()
partitions = get_table('t_dataset_partitions')
if not partitions.empty:
    df_dataset_datasources = partitions.copy()
    if 'query' in df_dataset_datasources.columns:
        df_dataset_datasources['datasource'] = df_dataset_datasources['query'].apply(parse_source_expression)
    elif 'source' in df_dataset_datasources.columns:
        df_dataset_datasources['datasource'] = df_dataset_datasources['source'].apply(parse_source_expression)
    else:
        df_dataset_datasources['datasource'] = None
    keep_columns = [c for c in ['table_name', 'partition_name', 'dataset_id', 'workspace_id', 'mode', 'source_type', 'datasource'] if c in df_dataset_datasources.columns]
    df_dataset_datasources = df_dataset_datasources[keep_columns].copy()
else:
    print('Skipping t_dataset_datasources build: t_dataset_partitions is empty')

write_delta_table(df_dataset_datasources, 't_dataset_datasources')
print(f"Created t_dataset_datasources with {len(df_dataset_datasources)} rows")
end_phase('build_dataset_datasources')

StatementMeta(, 04feeb51-f3ac-4f91-ad25-85e161dd4f5b, 8, Finished, Available, Finished, False)

Created t_dataset_datasources with 49 rows


In [ ]:
start_phase('build_edges')

edge_frames = []

dependency_timer_start = time.perf_counter()
dependencies = get_table('t_dataset_dependencies')
if not dependencies.empty:
    # Add foreign keys based on object types
    dep_edges = add_dependency_foreign_keys(dependencies)
    dep_edges['edge_id'] = dep_edges['dependency_pk']
    dep_edges['from_node'] = dep_edges['referenced_object_key']
    dep_edges['to_node'] = dep_edges['object_key']
    dep_edges['edge_type'] = dep_edges['object_type']
    dep_edges['dataset_id'] = dep_edges.get('dataset_id')
    dep_edges['workspace_id'] = dep_edges.get('workspace_id')
    edge_frames.append(dep_edges[['edge_id', 'from_node', 'to_node', 'edge_type', 'dataset_id', 'workspace_id']])
    print(f"Dependency edges: {len(dep_edges)} rows")
else:
    print('Skipping dependency edges: t_dataset_dependencies is empty')
print(f"[timer] edge_section:dependencies: {time.perf_counter() - dependency_timer_start:.2f}s")

report_timer_start = time.perf_counter()
report_objects = get_table('t_report_semantic_objects')
if not report_objects.empty:
    # Add foreign keys with dataset_id lookup from report_metadata
    report_edges = add_report_object_foreign_keys(report_objects, get_table('t_report_metadata'))
    report_edges['edge_id'] = report_edges['semantic_object_pk']
    report_edges['from_node'] = report_edges['object_key']
    report_edges['to_node'] = report_edges['visual_fk']
    report_edges['edge_type'] = report_edges['report_source']

    report_edges['dataset_id'] = report_edges.get('dataset_id')
    report_edges['workspace_id'] = report_edges.get('workspace_id')
    edge_frames.append(report_edges[['edge_id', 'from_node', 'to_node', 'edge_type',  'dataset_id', 'workspace_id']])
    print(f"Report semantic edges: {len(report_edges)} rows")
else:
    print('Skipping report semantic edges: t_report_semantic_objects is empty')
print(f"[timer] edge_section:report_semantic: {time.perf_counter() - report_timer_start:.2f}s")

# Direct Lake edges - connecting lakehouse tables to dataset tables via Direct Lake
directlake_timer_start = time.perf_counter()
direct_lake_sources = get_table('t_direct_lake_sources')
if not direct_lake_sources.empty:
    dataset_tables = get_table('t_dataset_tables')
    lakehouses_meta = get_table('t_lakehouses_meta')
    lakehouse_tables = get_table('t_lakehouse_tables')
    
    if not dataset_tables.empty and not lakehouses_meta.empty and not lakehouse_tables.empty:
        # Join: t_direct_lake_sources -> t_dataset_tables (on dataset_id)
        dl_edges = direct_lake_sources.merge(
            dataset_tables,
            on='dataset_id',
            how='inner',
            suffixes=('', '_dst')
        )
        
        # Join: -> t_lakehouses_meta (on workspace_id and itemname = lakehouse_name)
        dl_edges = dl_edges.merge(
            lakehouses_meta,
            left_on=['workspace_id', 'itemname'],
            right_on=['workspace_id', 'lakehouse_name'],
            how='inner',
            suffixes=('', '_lh')
        )
        
        # Join: -> t_lakehouse_tables (on lakehouse_id and dst.name = table_name)
        dl_edges = dl_edges.merge(
            lakehouse_tables,
            left_on=['lakehouse_id', 'name'],
            right_on=['lakehouse_id', 'table_name'],
            how='inner',
            suffixes=('', '_lht')
        )
        
        # Create edge columns
        dl_edges['edge_id'] = dl_edges['name'] + '-' + dl_edges['lakehouse_id'] + '-directlake'
        dl_edges['from_node'] = dl_edges['lakehouse_table_pk']
        dl_edges['to_node'] = dl_edges['table_pk']
        dl_edges['edge_type'] = 'Directlake Table'
        
        edge_frames.append(dl_edges[['edge_id', 'from_node', 'to_node', 'edge_type', 'dataset_id', 'workspace_id']])
        print(f"DirectLake edges: {len(dl_edges)} rows")
    else:
        print('Skipping DirectLake edges: one or more required tables are empty')
else:
    print('Skipping DirectLake edges: t_direct_lake_sources is empty')
print(f"[timer] edge_section:directlake: {time.perf_counter() - directlake_timer_start:.2f}s")

# Table Shortcut edges
shortcut_timer_start = time.perf_counter()
sc_edges = pd.DataFrame(columns=['edge_id', 'from_node', 'to_node', 'edge_type', 'dataset_id', 'workspace_id'])
shortcut_sources = get_table('t_lakehouse_shortcuts')
if not shortcut_sources.empty:
    sc_edges = pd.DataFrame()
    sc_edges['edge_id'] = shortcut_sources['shortcut_name'] + '-' + shortcut_sources['lakehouse_id']
    sc_edges['from_node'] = shortcut_sources['shortcut_name'] + '-' + shortcut_sources['source_item_id']
    sc_edges['to_node'] = shortcut_sources['shortcut_name'] + '-' + shortcut_sources['lakehouse_id']
    sc_edges['edge_type'] = 'Shortcut'
    sc_edges['dataset_id'] = None
    sc_edges['workspace_id'] = shortcut_sources['workspace_id']
    sc_edges = sc_edges.drop_duplicates()
    edge_frames.append(sc_edges[['edge_id', 'from_node', 'to_node', 'edge_type', 'dataset_id', 'workspace_id']])
    print(f"Shortcut edges: {len(sc_edges)} rows")
else:
    print('Skipping shortcut edges: t_lakehouse_shortcuts is empty')
print(f"[timer] edge_section:shortcuts: {time.perf_counter() - shortcut_timer_start:.2f}s")

df_dataset_edges = pd.concat(edge_frames, axis=0, ignore_index=True) if edge_frames else pd.DataFrame(columns=['edge_id', 'from_node', 'to_node', 'edge_type', 'dataset_id', 'workspace_id'])
write_delta_table(df_dataset_edges, 'v_edges')
print(f"Created v_edges with {len(df_dataset_edges)} rows")

end_phase('build_edges')

StatementMeta(, 04feeb51-f3ac-4f91-ad25-85e161dd4f5b, 9, Finished, Available, Finished, False)

Created v_edges with 1238 rows


In [6]:
display(sc_edges)

StatementMeta(, 04feeb51-f3ac-4f91-ad25-85e161dd4f5b, 10, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, b26e0970-092c-4382-93ae-669c51a91361)

In [7]:
notebookutils.session.stop()

StatementMeta(, 04feeb51-f3ac-4f91-ad25-85e161dd4f5b, 11, Finished, Available, Finished, False)